# Multi-slice spatial niche identification and niche-associated gene analysis

This notebook applies **IDEA** to jointly identify spatial niches across six Xenium kidney slices collected at different time points.

The workflow includes:

1. importing the required packages;
2. specifying the input Xenium datasets and sample names;
3. initializing the conditional multi-slice niche identification model;
4. training the model to learn shared spatial representations;
5. predicting spatial niches and saving the annotated datasets;
6. identifying niche-associated genes using model attribution; and
7. evaluating niche-associated genes by differential expression analysis.


## 1. Import required packages

`scanpy` is used for handling spatial transcriptomics data stored as `AnnData` objects, while `init_niche_model` initializes the **IDEA** model for spatial niche identification.


In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import scanpy as sc
from IDEA import init_niche_model

## 2. Specify the Xenium kidney datasets and sample names

Six Xenium kidney slices are included in the joint analysis:

- `ShamL`
- `Hour4L`
- `Hour12L`
- `Day2L`
- `Day14L`
- `week6L`

`paths` contains the input `.h5ad` files, and `sample_names` provides the corresponding sample identifiers used during training, prediction, and output.

The order of `sample_names` must match the order of the datasets in `paths`.


In [2]:

paths = [
    "/home/ren_jiale/Xenium_kidney/Xenium_Day2L.h5ad",
    "/home/ren_jiale/Xenium_kidney/Xenium_Day14L.h5ad",
    "/home/ren_jiale/Xenium_kidney/Xenium_Hour4L.h5ad",
    "/home/ren_jiale/Xenium_kidney/Xenium_Hour12L.h5ad",
    "/home/ren_jiale/Xenium_kidney/Xenium_ShamL.h5ad",
    "/home/ren_jiale/Xenium_kidney/Xenium_week6L.h5ad",
]

sample_names = [
    "Day2L",
    "Day14L",
    "Hour4L",
    "Hour12L",
    "ShamL",
    "week6L",
]

## 3. Initialize the IDEA-N model

`init_niche_model` initializes IDEA-N for joint niche identification across multiple Xenium kidney slices.

Key settings used here include:

- `st_ad=paths`: input spatial transcriptomics datasets.
- `sample_names=sample_names`: identifiers corresponding to the input slices.
- `use_condition=True`: enables conditional modeling so that slice-associated variation can be modeled while learning a shared latent representation.
- `output_dir="./results"`: directory used to store model outputs.
- `target=4096`, `min_spots=500`, `max_spots=6000`: control the target, minimum, and maximum sizes of local spatial blocks.
- `sliding=True`: enables sliding spatial partitioning.
- `hidden_size=256`, `latent_size=128`: dimensions of the hidden and latent representations.
- `spatial_weight=5`: weight of the spatial regularization objective.
- `radius_scale=5`: controls the spatial neighborhood radius.
- `lr_encoder`, `lr_decoder`, `lr_classifier`: learning rates for the corresponding model components.
- `seed=42`: random seed used for reproducibility.

The block-wise design restricts spatial modeling to local neighborhoods, avoiding construction of a single global spatial graph across all slices.



In [3]:
model = init_niche_model(
        st_ad=paths,
        sample_names=sample_names,

        use_condition=True,
        output_dir="./results",
        target=4096,
        min_spots=500,
        max_spots=6000,
        sliding=True,
        hidden_size=256,
        latent_size=128,

        spatial_weight=5,
        radius_scale=5,

        lr_encoder=0.0001,
        lr_decoder=0.0001,
        lr_classifier=0.0005,
        seed=42,
    )

[IDEA-N] Path-streaming mode enabled: 6 section(s). Only one complete h5ad is kept in RAM at a time.
[IDEA-N] Reading gene metadata for Day2L...
[IDEA-N] Reading gene metadata for Day14L...
[IDEA-N] Reading gene metadata for Hour4L...
[IDEA-N] Reading gene metadata for Hour12L...
[IDEA-N] Reading gene metadata for ShamL...
[IDEA-N] Reading gene metadata for week6L...
[IDEA-N] Number of common genes: 299
[IDEA-N] Selecting HVGs for Day2L...
[IDEA-N] Day2L: 299 HVGs
[IDEA-N] Selecting HVGs for Day14L...
[IDEA-N] Day14L: 299 HVGs
[IDEA-N] Selecting HVGs for Hour4L...
[IDEA-N] Hour4L: 299 HVGs
[IDEA-N] Selecting HVGs for Hour12L...
[IDEA-N] Hour12L: 299 HVGs
[IDEA-N] Selecting HVGs for ShamL...
[IDEA-N] ShamL: 299 HVGs
[IDEA-N] Selecting HVGs for week6L...
[IDEA-N] week6L: 299 HVGs
[IDEA-N] Final HVG union: 299 genes
[IDEA-N] Preparing Day2L...
[IDEA-N] Preparing Day14L...
[IDEA-N] Preparing Hour4L...
[IDEA-N] Preparing Hour12L...
[IDEA-N] Preparing ShamL...
[IDEA-N] Preparing week6L...


## 4. Train the multi-slice niche identification model

IDEA-N is trained in two stages.

- **Stage 1** learns a spatially informed latent representation.
- **Stage 2** trains the niche predictor using clustering-derived pseudo-labels.

Here, Stage 1 is trained for `20` epochs and Stage 2 for `50` epochs.

`n_clusters=None` allows the number of niches to be determined from Leiden clustering. Leiden clustering is performed with `resolution=0.15` and `25` neighbors.

The training history is exported to `./results/training_history.csv`.


In [4]:
history = model.train_model(
    stage1_epochs=20,
    stage2_epochs=50,
    n_clusters=None,
    leiden_resolution=0.15,
    leiden_neighbors=25,
)
history.to_csv(f"./results/training_history.csv", index=False)

[IDEA-N stage2] 50/50: 100%|██████████| 195/195 [00:02<00:00, 65.54it/s, class_loss=0.0161]


## 5. Predict spatial niches and save the results

After training, `model.predict(obs_key="niche")` predicts a niche label for each spatial unit and stores the categorical assignments under `obs["niche"]`.

The outputs include:

- `niche_predictions.csv`: predicted niche assignments across all six slices;
- one compressed `.h5ad` file for each slice containing the inferred niche labels; and
- `IDEA_conditional.pt`: the trained conditional IDEA-N model.

These annotated `AnnData` objects can be directly reused for downstream spatial visualization and biological analysis.


In [5]:
pred = model.predict(obs_key="niche")
pred.to_csv(f"./results/niche_predictions.csv")

for name, adata in zip(model.metadata["sample_names"], model.adatas):
    adata.write_h5ad(f"./results/{name}_with_niche.h5ad", compression="gzip", compression_opts=4)

model.save("./results/IDEA_conditional.pt")

[IDEA-N] embedding: 100%|██████████| 195/195 [00:03<00:00, 62.23it/s]


## 6. Identify niche-associated genes by model attribution

The trained model is further interpreted to identify genes associated with its niche predictions.

`model.explain()` computes post-hoc feature attributions for the inferred niches:

- `top_n=30`: retains the top 30 attributed genes for each target niche;
- `n_steps=50`: number of interpolation steps used for attribution;
- `attribution_scope="all"`: performs attribution across all analyzed spatial units; and
- `output_dir`: directory used to save the interpretation results.

These attribution scores link the inferred spatial niches to molecular features that contribute directly to the model predictions.


In [6]:
result = model.explain(
    top_n=30,
    n_steps=50,
    output_dir="./results/niche_interpretation",
    attribution_scope="all",
)

[IDEA-I niche] blocks: 100%|██████████| 195/195 [03:38<00:00,  1.12s/it]


## 7. Evaluate niche-associated genes by differential expression

Candidate genes identified by model attribution are further evaluated using differential expression analysis.

`model.calculate_de()` uses the attributed genes stored in `result["genes"]` and compares their expression across the inferred niches.

Key settings include:

- `mode="joint"`: performs the analysis jointly across the integrated slices.
- `best_only=True`: retains the best-supported niche association for each candidate gene.
- `output_path`: saves the resulting differential expression table to `./results/niche_batch/niche_DE_joint.csv`.

This step provides an additional expression-based assessment of the genes highlighted by model attribution.

In [7]:
de_df = model.calculate_de(
    high_genes=result['genes'],
    mode="joint",
    best_only=True,
    output_path="./results/niche_batch/niche_DE_joint.csv",
)